# ScatterBrain — Basic Workflow Tutorial

This notebook demonstrates the complete Phase 1 MVP workflow:

1. Load 1D ASCII scattering data
2. Inspect and plot the raw curve
3. Subtract a constant background
4. Guinier analysis → Rg and I(0)
5. Porod analysis → exponent and Porod constant
6. Fit a sphere form factor model
7. Save the processed curve

The example data represents a monodisperse sphere system with  
R ≈ 3.87 nm (Rg ≈ 3 nm), I(0) ≈ 1×10⁵, and a small constant background of ~10 counts.

In [ ]:
import pathlib
import numpy as np
import matplotlib
matplotlib.use('Agg')  # use non-interactive backend; remove for interactive sessions
import matplotlib.pyplot as plt

import scatterbrain
from scatterbrain.io import load_ascii_1d, save_ascii_1d
from scatterbrain.processing import subtract_background
from scatterbrain.analysis.guinier import guinier_fit
from scatterbrain.analysis.porod import porod_analysis
from scatterbrain.modeling.form_factors import sphere_pq
from scatterbrain.modeling.fitting import fit_model
from scatterbrain.visualization import plot_iq, plot_guinier, plot_porod, plot_fit

print(f"ScatterBrain version: {scatterbrain.__version__}")

## 1. Load data

In [ ]:
# Locate the bundled example file
data_file = pathlib.Path(scatterbrain.__file__).parent / "examples" / "data" / "example_sphere_data.dat"
print(f"Loading: {data_file}")

curve = load_ascii_1d(
    data_file,
    q_col=0,
    i_col=1,
    err_col=2,
    skip_header=5,          # skip the 5 comment lines
    delimiter=r'\s+',
)
print(curve)

## 2. Inspect and plot the raw curve

In [ ]:
fig, ax = plot_iq(curve, title="Raw SAXS data — Sphere example", labels="Raw data")
plt.savefig("raw_iq.png", dpi=100)
plt.show()
print(f"q range : {curve.q.min():.3f} – {curve.q.max():.3f} {curve.q_unit}")
print(f"I range : {curve.intensity.min():.3e} – {curve.intensity.max():.3e} {curve.intensity_unit}")
print(f"Points  : {len(curve)}")

## 3. Subtract a constant background

The data header notes a background of ~10 counts.

In [ ]:
BACKGROUND = 10.0
curve_bg = subtract_background(curve, BACKGROUND)

fig, ax = plot_iq(
    [curve, curve_bg],
    labels=["Raw", f"Background subtracted (−{BACKGROUND})"],
    title="Background subtraction",
)
plt.savefig("bg_subtracted.png", dpi=100)
plt.show()
print("Processing history:", curve_bg.metadata["processing_history"])

## 4. Guinier analysis

The Guinier approximation: $\ln I(q) = \ln I(0) - \frac{R_g^2}{3} q^2$  
Valid for $qR_g \lesssim 1.3$.

In [ ]:
g_result = guinier_fit(curve_bg, qrg_limit_max=1.3)

if g_result is not None:
    print(f"Rg  = {g_result['Rg']:.4f} ± {g_result['Rg_err']:.4f} nm⁻¹")
    print(f"I(0)= {g_result['I0']:.4e} ± {g_result['I0_err']:.4e} {curve_bg.intensity_unit}")
    print(f"R²  = {g_result['r_value']**2:.5f}")
    print(f"Fit q-range: {g_result['q_fit_min']:.4f} – {g_result['q_fit_max']:.4f} {curve_bg.q_unit}")
    print(f"Points in fit: {g_result['num_points_fit']}")
else:
    print("Guinier fit failed — check data quality or q-range.")

In [ ]:
fig, ax = plot_guinier(curve_bg, guinier_result=g_result, title="Guinier Plot")
plt.savefig("guinier.png", dpi=100)
plt.show()

## 5. Porod analysis

For smooth 3-D interfaces: $I(q) \propto K_P\, q^{-4}$  
A log-log fit of $I(q)$ vs $q$ at high $q$ yields the Porod exponent $n$ and constant $K_P$.

In [ ]:
p_result = porod_analysis(curve_bg, q_fraction_high=0.3, fit_log_log=True)

if p_result is not None:
    print(f"Porod exponent n  = {p_result['porod_exponent']:.4f} ± {p_result['porod_exponent_err']:.4f}")
    print(f"Porod constant Kp = {p_result['porod_constant_kp']:.4e} ± {p_result['porod_constant_kp_err']:.4e}")
    print(f"R² (log-log fit)  = {p_result['r_value']**2:.5f}")
    print(f"Fit q-range: {p_result['q_fit_min']:.4f} – {p_result['q_fit_max']:.4f} {curve_bg.q_unit}")
else:
    print("Porod analysis failed.")

In [ ]:
fig1, ax1 = plot_porod(curve_bg, porod_result=p_result,
                       plot_type="Iq4_vs_q", title="Porod Plot — I(q)·q⁴ vs q")
plt.savefig("porod_Iq4.png", dpi=100)
plt.show()

fig2, ax2 = plot_porod(curve_bg, porod_result=p_result,
                       plot_type="logI_vs_logq", title="Porod Plot — log(I) vs log(q)")
plt.savefig("porod_loglog.png", dpi=100)
plt.show()

## 6. Fit a sphere form factor

The model is:
$$I(q) = \text{scale} \cdot P_{\text{sphere}}(q, R) + \text{background}$$

Initial guesses from the Guinier result: $R \approx \sqrt{5/3}\,R_g$.

In [ ]:
# Initial radius guess from Guinier Rg (for a sphere, R = sqrt(5/3) * Rg)
rg_est = g_result["Rg"] if g_result is not None else 3.0
r_init = np.sqrt(5.0 / 3.0) * rg_est
scale_init = curve_bg.intensity.max()   # rough scale = I(0)
print(f"Initial guesses: scale={scale_init:.3e}, background=0, R={r_init:.3f} nm⁻¹")

fit_result = fit_model(
    curve_bg,
    model_func=sphere_pq,
    param_names=["radius"],
    initial_params=[scale_init, 0.0, r_init],        # [scale, background, radius]
    param_bounds=([0, -np.inf, 0.1], [np.inf, np.inf, 50.0]),
)

if fit_result is not None:
    fp = fit_result["fitted_params"]
    fe = fit_result["fitted_params_stderr"]
    print(f"\nFit results:")
    print(f"  scale      = {fp['scale']:.4e} ± {fe['scale']:.4e}")
    print(f"  background = {fp['background']:.4e} ± {fe['background']:.4e}")
    print(f"  radius     = {fp['radius']:.4f} ± {fe['radius']:.4f} nm⁻¹")
    print(f"  χ²_red     = {fit_result['chi_squared_reduced']:.4f}")
    print(f"  Fit success: {fit_result['success']}")
else:
    print("Fit did not converge.")

In [ ]:
if fit_result is not None:
    fig = plot_fit(
        curve_bg,
        fit_result,
        plot_residuals=True,
        title="Sphere Form Factor Fit",
    )
    plt.savefig("sphere_fit.png", dpi=100)
    plt.show()

## 7. Save the processed curve

In [ ]:
out_path = pathlib.Path("processed_sphere.dat")
save_ascii_1d(
    curve_bg,
    out_path,
    include_error=True,
    delimiter="\t",
    header="Background-subtracted sphere data — ScatterBrain tutorial",
)
print(f"Saved to {out_path.resolve()}")

# Round-trip check
curve_reloaded = load_ascii_1d(out_path, err_col=2, delimiter=r'\s+')
print(f"Reloaded {len(curve_reloaded)} points — q matches: ",
      np.allclose(curve_bg.q, curve_reloaded.q, rtol=1e-5))

## Summary

| Quantity | Value |
|----------|-------|
| Rg (Guinier) | see above |
| I(0) (Guinier) | see above |
| Porod exponent n | see above |
| Sphere radius R (fit) | see above |
| χ²_red | see above |

**Next steps (Phase 2):** automatic q-range selection refinement, additional form factors (cylinder, core-shell), p(r) via IFT, and interactive plots.